# Synthèse transversale finale
### Comment le deep learning adapte ses architectures à la structure des données

**Projet de fin de module — Deep Learning — EMSI Casablanca, 2025–2026**

---

Ce notebook conclut le projet en reliant les trois parties (MLP, CNN, modèles
séquentiels). Il répond à la **question transversale** du cahier des charges :

> *Comment le deep learning adapte-t-il ses architectures à la structure des
> données — tabulaire, image et séquentielle — et pourquoi un même paradigme
> d'apprentissage supervisé doit-il être décliné différemment selon la
> géométrie, la dépendance locale, la temporalité et la représentation des
> données ?*


## 1. Un même paradigme, trois biais inductifs

Les trois parties partagent **exactement le même paradigme d'apprentissage
supervisé** : un modèle paramétré $f_\theta$, une fonction de perte différentiable,
une descente de gradient par rétropropagation, une séparation train/val/test, et
une évaluation par des métriques adaptées. Ce qui change d'une partie à l'autre
n'est pas l'optimisation, mais le **biais inductif** : l'hypothèse structurelle
que l'architecture impose pour **coller à la géométrie des données**.

| Critère | Partie I — MLP | Partie II — CNN | Partie III — RNN/Seq2Seq |
|---------|----------------|-----------------|--------------------------|
| Type de données | Tabulaire | Image (grille 2D) | Séquence (texte) |
| Structure exploitée | Aucune (attributs interchangeables) | Localité spatiale, invariance par translation | Ordre, dépendance temporelle |
| Biais inductif | Connexions denses | Partage de poids + champs réceptifs locaux | État caché récurrent (mémoire) |
| Brique clé | `nn.Linear` | `nn.Conv2d` + pooling | `nn.RNN/LSTM/GRU` |
| Paramètres / entrée | Élevé (tout connecté) | Faible (noyaux partagés) | Partagés dans le temps |
| Dataset | Breast Cancer Wisconsin | Fashion-MNIST | fra-eng (Tatoeba) |
| Métriques | Acc, P, R, F1, AUC | Accuracy, confusion | Perplexité, BLEU |


## 2. Pourquoi décliner l'architecture selon les données ?

**Géométrie (tabulaire → MLP).** Les colonnes d'une table n'ont pas de
voisinage privilégié : permuter deux attributs ne change pas le problème. Le MLP,
qui traite toutes les entrées symétriquement via des couches denses, est donc le
biais minimal adéquat. Imposer de la localité (CNN) ou de la récurrence (RNN)
ici serait une hypothèse **fausse** et contre-productive.

**Dépendance locale (image → CNN).** Dans une image, l'information est portée
par des **motifs locaux** (bords, textures) invariants par translation. Le CNN
encode cet *a priori* par le **partage des poids** et des **champs réceptifs
locaux** : il atteint de meilleures performances qu'un MLP avec **moins de
paramètres** (vérifié en Partie II), parce qu'il ne réapprend pas le même motif
à chaque position.

**Temporalité (séquence → RNN/Seq2Seq).** Dans une phrase, l'ordre est
constitutif du sens et les longueurs varient. Le RNN encode cette **dépendance
temporelle** via un état caché transmis pas à pas ; LSTM/GRU stabilisent la
mémoire longue (vérifié par la perplexité en Partie III). Quand entrée et sortie
sont deux séquences non alignées (traduction), le schéma **encodeur–décodeur**
devient nécessaire.

**Représentation.** Le prétraitement suit la même logique : standardisation
pour le tabulaire, normalisation des pixels pour l'image, tokenisation +
vocabulaire + padding/masquage pour le texte. La **représentation d'entrée** est
elle aussi adaptée à la structure des données.


In [ ]:
# Récapitulatif programmatique : nombre de paramètres pour traiter une même
# "taille d'entrée" selon le biais architectural (illustration conceptuelle).
import importlib, subprocess, sys
try: import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=False); import torch
from torch import nn

# Entrée fictive type image 1x28x28 = 784 valeurs.
mlp_layer = nn.Linear(28 * 28, 64)                       # dense
cnn_layer = nn.Conv2d(1, 64, kernel_size=5, padding=2)   # convolution
rnn_layer = nn.RNN(28, 64, batch_first=True)             # 28 pas de longueur, dim 28

def n_params(m): return sum(p.numel() for p in m.parameters())
print("Pour produire 64 canaux/neurones à partir d'une entrée 28x28 :")
print(f"  MLP (Linear 784->64)         : {n_params(mlp_layer):>7,} paramètres")
print(f"  CNN (Conv2d 1->64, k=5)       : {n_params(cnn_layer):>7,} paramètres")
print(f"  RNN (séquence de 28, dim 28)  : {n_params(rnn_layer):>7,} paramètres")
print("\nLe partage de poids (CNN/RNN) réduit drastiquement les paramètres "
      "tout en exploitant la structure des données.")

## 3. Continuité conceptuelle et conclusion

Les trois familles ne sont pas des outils isolés mais une **progression
cohérente** : on part de la connexion dense universelle (MLP), on y injecte un
*a priori* spatial (CNN), puis un *a priori* temporel (RNN), et enfin un cadre
conditionnel entrée→sortie (Seq2Seq). À chaque étape, on **contraint**
l'espace des fonctions apprenables pour qu'il corresponde à la structure réelle
des données — c'est précisément ce qui permet d'apprendre efficacement avec
moins de données et moins de paramètres.

**Réponse synthétique.** Le deep learning adapte ses architectures en
choisissant, pour chaque type de données, le **biais inductif** qui reflète sa
structure : symétrie des attributs pour le tabulaire (MLP), localité et
invariance pour l'image (CNN), ordre et mémoire pour la séquence (RNN/Seq2Seq).
Le paradigme d'apprentissage supervisé — perte différentiable + rétropropagation
— reste **identique** ; seule change la **forme de la fonction** $f_\theta$, parce
que la géométrie, la dépendance locale, la temporalité et la représentation des
données imposent des hypothèses structurelles différentes. Adapter
l'architecture à la donnée, c'est donc **encoder la bonne hypothèse a priori**,
ce que nos trois études expérimentales valident de bout en bout.

---

*Fin du projet — l'ensemble des livrables (théorie, code commenté, expériences,
figures, métriques et analyses critiques) est contenu dans les quatre notebooks
de ce dépôt.*
